### Notebook 3: Data Balancing, Scaling & Feature Selection
**Objective:** Perform a 3-way Stratified Split (Train / Validation / Test), scale continuous features, apply SMOTE to training data, select top features, and export datasets.

In [11]:
%pip install imbalanced-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTENC
from sklearn.ensemble import RandomForestClassifier

# 1. Load cleaned dataset
df = pd.read_csv("../data/telco_cleaned.csv")

# 2. Separate Features and Target
X = df.drop(columns=['Churn'])
y = df['Churn']

# 3. Step 1 Split: 70% Train, 30% Temporary (Validation + Test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# 4. Step 2 Split: Split Temporary 50/50 -> 15% Validation, 15% Test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

# 5. Fit Scaler ONLY on Training Data (prevents data leakage)
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
scaler = StandardScaler()

X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_val[num_cols] = scaler.transform(X_val[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

print(f"Train Shape: {X_train.shape} | Val Shape: {X_val.shape} | Test Shape: {X_test.shape}")

Train Shape: (4930, 30) | Val Shape: (1056, 30) | Test Shape: (1057, 30)


In [13]:
# Identify categorical feature indices — everything except the three numeric columns

num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
categorical_features = [i for i, col in enumerate(X_train.columns) if col not in num_cols]

# Apply SMOTENC oversampling ONLY to X_train and y_train
smotenc = SMOTENC(categorical_features=categorical_features, random_state=42)
X_train_res, y_train_res = smotenc.fit_resample(X_train, y_train)

print("Pre-SMOTE Training Distribution:")
print(y_train.value_counts())
print("\nPost-SMOTE Training Distribution:")
print(y_train_res.value_counts())
print(f"\nValidation set remains untouched (Real ratio): {dict(y_val.value_counts())}")
print(f"Test set remains untouched (Real ratio): {dict(y_test.value_counts())}")

Pre-SMOTE Training Distribution:
Churn
0    3622
1    1308
Name: count, dtype: int64

Post-SMOTE Training Distribution:
Churn
0    3622
1    3622
Name: count, dtype: int64

Validation set remains untouched (Real ratio): {0: np.int64(776), 1: np.int64(280)}
Test set remains untouched (Real ratio): {0: np.int64(776), 1: np.int64(281)}


In [14]:
from sklearn.feature_selection import mutual_info_classif

# Compute Mutual Information between each feature and the target

# MI works cleanly on both the one-hot categorical columns and the

# scaled continuous columns, and doesn't assume a linear relationship

mi_scores = mutual_info_classif(X_train_res, y_train_res, random_state=42)

importance_df = pd.DataFrame({

    'Feature': X_train_res.columns,

    'Importance': mi_scores

}).sort_values(by='Importance', ascending=False)

# Select top 15 features across Train, Val, and Test

top_features = importance_df.head(15)['Feature'].tolist()

X_train_final = X_train_res[top_features]

X_val_final = X_val[top_features]

X_test_final = X_test[top_features]

print(f"Selected Top {len(top_features)} Features (by Mutual Information):")

print(top_features)

Selected Top 15 Features (by Mutual Information):
['tenure', 'MonthlyCharges', 'Contract_Two year', 'PaymentMethod_Electronic check', 'InternetService_Fiber optic', 'DeviceProtection_No internet service', 'TotalCharges', 'StreamingMovies_No internet service', 'OnlineBackup_No internet service', 'TechSupport_No internet service', 'StreamingTV_No internet service', 'OnlineSecurity_No internet service', 'OnlineSecurity_Yes', 'InternetService_No', 'Dependents_Yes']


In [15]:
# Save final balanced and feature-selected splits to data/
os.makedirs("../data", exist_ok=True)

X_train_final.to_csv("../data/X_train.csv", index=False)
X_val_final.to_csv("../data/X_val.csv", index=False)
X_test_final.to_csv("../data/X_test.csv", index=False)

y_train_res.to_csv("../data/y_train.csv", index=False)
y_val.to_csv("../data/y_val.csv", index=False)
y_test.to_csv("../data/y_test.csv", index=False)

print("Successfully exported X_train, X_val, X_test, y_train, y_val, and y_test to '../data/'!")

Successfully exported X_train, X_val, X_test, y_train, y_val, and y_test to '../data/'!
